# Data Collection

This notebook downloads the [Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) dataset from the UCI Machine Learning Repository and saves it to S3 as a parquet file. The dataset contains transactional data from a UK-based online retail company (Dec 2009 - Dec 2011) and will be used for customer segmentation via cluster analysis.

## Imports

In [ ]:
import pandas as pd
import zipfile
import io
import requests
import pyarrow
import s3fs

## Constants

In [ ]:
str_bucket = 'cluster-analysis-demo'
str_task = '00_data_collection'
str_url = 'https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip'

## Download Data

In [ ]:
# download zip file
response = requests.get(str_url)
response.raise_for_status()

# extract xlsx from zip
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # find the xlsx file
    list_files = z.namelist()
    print(f'Files in zip: {list_files}')
    # get xlsx filename
    str_xlsx = [f for f in list_files if f.endswith('.xlsx')][0]
    # read xlsx
    with z.open(str_xlsx) as f:
        df = pd.read_excel(f, engine='openpyxl')

## Inspect Data

In [ ]:
# shape
print(f'Shape: {df.shape}')

# data types
print(f'\nData Types:\n{df.dtypes}')

# first 5 rows
df.head()

In [ ]:
# missing values
print(f'Missing Values:\n{df.isnull().sum()}')
print(f'\nMissing Value Proportion: {df.isnull().sum().sum() / (df.shape[0] * df.shape[1]):.4f}')

In [ ]:
# unique customers
print(f'Unique Invoices: {df["Invoice"].nunique():,}')
print(f'Unique Customers: {df["Customer ID"].nunique():,}')
print(f'Unique Products: {df["StockCode"].nunique():,}')
print(f'Unique Countries: {df["Country"].nunique():,}')
print(f'\nDate Range: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')

## Clean Column Names

In [ ]:
# clean column names (snake_case)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w]', '', regex=True)
)

# check
print(f'Columns: {df.columns.tolist()}')

## Save to S3

In [ ]:
# save to s3 as parquet
str_uri = f's3://{str_bucket}/{str_task}/data.parquet'
df.to_parquet(str_uri, index=False)
print(f'Saved {df.shape[0]:,} rows to {str_uri}')